In [ ]:
## Getting Fit Result

Before we process anything, we have to first readthe fitdata

In [ ]:
fit_res=eispac.read_fit('data_eis/eis_20140202_102527.fe_12_195_119.2c-0.fit.h5')

In [ ]:
fit_wave_cube, fit_inten_cube = fit_res.get_fit_profile(component=[0,1])
sum_fit_inten = fit_inten_cube.sum(axis=2)
fit_x, fit_y = fit_res.get_fit_profile(component=0)


In [1]:
import matplotlib.pyplot as plt
import astropy.units as u
import eispac

if __name__ == '__main__':
    # Read in the fit template and EIS observation
    data_filepath = 'data_eis/eis_20140201_132637.data.h5'
    template_filepath = eispac.data.get_fit_template_filepath("fe_12_195_119.2c.template.h5")
    tmplt = eispac.read_template(template_filepath)
    data_cube = eispac.read_cube(data_filepath, tmplt.central_wave)

    # Select a cutout of the raster (note the order of array & plot indices!)
    cutout_extent = [48, 165, 254, 378] # units of [arcsec]
    w_coords = data_cube.axis_world_coords('em.wl')
    lower_left = (cutout_extent[2]*u.arcsec, cutout_extent[0]*u.arcsec,
                  w_coords[0])
    upper_right = (cutout_extent[3]*u.arcsec, cutout_extent[1]*u.arcsec,
                   w_coords[-1])
    raster_cutout = data_cube.crop_by_coords(lower_left,
                                             upper_corner=upper_right)

    # Fit the data and save it to disk
    fit_res = eispac.fit_spectra(raster_cutout, tmplt, ncpu=2)
    save_filepaths = eispac.save_fit(fit_res, save_dir='cwd')

    # Extract array of total data and fit intensites
    sum_data_inten = raster_cutout.sum_spectra().data
    fit_wave_cube, fit_inten_cube = fit_res.get_fit_profile(component=[0,1])
    sum_fit_inten = fit_inten_cube.sum(axis=2)

    # Extract example fit profiles at a higher spectral resolution
    ex_coords = [43, 28] # [Y,X] array coords in units of [pixels]
    fit_x, fit_y = fit_res.get_fit_profile(coords=ex_coords,
                                           num_wavelengths=100)
    c0_fit_x, c0_fit_y = fit_res.get_fit_profile(component=0,
                                     coords=ex_coords, num_wavelengths=100)
    c1_fit_x, c1_fit_y = fit_res.get_fit_profile(component=1,
                                     coords=ex_coords, num_wavelengths=100)
    c2_fit_x, c2_fit_y = fit_res.get_fit_profile(component=2,
                                     coords=ex_coords, num_wavelengths=100)
    sub_data = raster_cutout.data[ex_coords[0], ex_coords[1], :]
    sub_wave = raster_cutout.wavelength[ex_coords[0], ex_coords[1], :]
    sub_err = raster_cutout.uncertainty.array[ex_coords[0], ex_coords[1], :]

    # Make a multi-panel figure with the cutout and example
    fig = plt.figure()
    plot_grid = fig.add_gridspec(nrows=2, ncols=2, hspace=0.5, wspace=0.3)

    data_img = fig.add_subplot(plot_grid[0,0])
    data_img.imshow(sum_data_inten, origin='lower', extent=cutout_extent,
                       cmap='gray')
    data_img.set_title('Data Cutout')
    data_img.set_xlabel('Solar-X [arcsec]')
    data_img.set_ylabel('Solar-Y [arcsec]')

    fit_img = fig.add_subplot(plot_grid[0,1])
    fit_img.imshow(sum_fit_inten, origin='lower', extent=cutout_extent,
                      cmap='gray')
    fit_img.set_title('Total Fit Intensity')
    fit_img.set_xlabel('Solar-X [arcsec]')
    fit_img.set_ylabel('Solar-Y [arcsec]')

    profile = fig.add_subplot(plot_grid[1,:])
    profile.errorbar(sub_wave, sub_data, yerr=sub_err,
                           ls='', marker='o', color='k')
    profile.plot(fit_x, fit_y, color='b', label='Combined profile')
    profile.plot(c0_fit_x, c0_fit_y, color='r', label='Gaussian 1')
    profile.plot(c1_fit_x, c1_fit_y, color='r', ls='--', label='Gaussian 2')
    profile.plot(c2_fit_x, c2_fit_y, color='g', label='Background')
    profile.set_title(f'Cutout indices iy = {ex_coords[0]},'
                            +f' ix = {ex_coords[1]}')
    profile.set_xlabel('Wavelength [$\AA$]')
    profile.set_ylabel('Intensity ['+str(raster_cutout.unit)+']')
    profile.legend(loc='upper left')
    plt.show()

Data file,
   /Users/ato/scripts/python_code/asheis/data_eis/eis_20140201_132637.data.h5
Header file,
   /Users/ato/scripts/python_code/asheis/data_eis/eis_20140201_132637.head.h5
Found a wavelength 195.11 [Angstroms] in window 8
INFO: uncertainty should have attribute uncertainty_type. [astropy.nddata.nddata]
 + computing fits for 0 exposures, each with 0 spectra
 + running mpfit on 0 cores (of 8)


ValueError: Number of processes must be at least 1

## Additional: PFSSpy Field Extrapolation

In [ ]:
from sunpy.net import Fido,attrs as a
import astropy.units as u
import sunpy.image.coalignment

# Defining the time of the AIA data we want
time_range = a.Time(map_eis.date_start-12*u.minute,
                    end=map_eis.date_end,
                    near=map_eis.date_average)

# Now we query hmi data
hmi_query = time_range & a.Instrument.hmi & a.Physobs.los_magnetic_field

In [ ]:
q = Fido.search(hmi_query)
files = Fido.fetch(q, path='data/')

In [ ]:
m_hmi = sunpy.map.Map(files)

In [ ]:
from astropy.coordinates import SkyCoord
import astropy.wcs
from reproject import reproject_interp
import numpy as np
import pfsspy
from astropy.visualization import ImageNormalize, quantity_support

shape_out=[540, 1080]
frame_out = SkyCoord(0, 0, unit=u.deg, rsun=m_hmi.coordinate_frame.rsun, frame="heliographic_stonyhurst", obstime=m_hmi.date)
header = sunpy.map.make_fitswcs_header(
    shape_out,
    frame_out,
    scale=[180 / shape_out[0], 360 / shape_out[1]] * u.deg / u.pix,
    projection_code="CAR",
)
out_wcs = astropy.wcs.WCS(header)
array, _ = reproject_interp(m_hmi, out_wcs, shape_out=shape_out)
array = np.where(np.isnan(array), 0, array)
m_hmi_cea = pfsspy.utils.car_to_cea(sunpy.map.Map(array, header))
m_hmi_cea.meta['TELESCOP'] = m_hmi.meta['TELESCOP']
m_hmi_cea.meta['CONTENT'] = 'Carrington Synoptic Chart Of Br Field'
m_hmi_cea.meta['T_OBS'] = m_hmi_cea.meta.pop('DATE-OBS')  # This is because of a bug where the date accidentally returns None if it is in the date-obs key
m_hmi_cea = sunpy.map.Map(
    m_hmi_cea.data,
    m_hmi_cea.meta,
    plot_settings={'norm': ImageNormalize(vmin=-1.5e3,vmax=1.5e3), 'cmap': 'hmimag'}
)

In [ ]:
nrho = 70
rss = 2.5
pfss_input = pfsspy.Input(m_hmi_cea, nrho, rss)
output = pfsspy.pfss(pfss_input)
m_hmi_cea.plot_settings = {'norm': ImageNormalize(vmin=-1.5e3,vmax=1.5e3), 'cmap': 'hmimag'}

In [ ]:
masked_pix_y, masked_pix_x = np.where(m_hmi_cea.data < -3e1)
seeds = m_hmi_cea.pixel_to_world(masked_pix_x*u.pix, masked_pix_y*u.pix,).make_3d()

In [ ]:
seeds = m_hmi_cea.pixel_to_world(masked_pix_x*u.pix, masked_pix_y*u.pix,).make_3d()